# 04 — Waitlist, Access & The Workforce Crisis

**Goal:** Connect the causal chain: low wages → workforce shortage → providers can't staff →
waitlist grows → 100,000+ Texans wait years for services.

**Data Sources:**
- HHSC Interest List Reduction reports (hhs.texas.gov)
- LBB rider packets: slots funded and waitlist size by biennium (83rd-89th Legislatures)
- NCI Staff Stability Survey: national DSP turnover rates
- PHI (Paraprofessional Healthcare Institute): state-level DSP workforce data
- Texas Tribune (2024-06-17): "decade-long waits"
- KERA (2025-03-04): "$12/hr not enough for disability services"

**Key Questions:**
1. How has the HCS interest list changed over the last decade?
2. At current funding pace, how long to serve everyone waiting?
3. What does 50% annual turnover cost providers?
4. How does the workforce crisis directly cause the access crisis?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import sys
sys.path.insert(0, '../src')

from texas_hhcs.budget import BienniumBudget, build_trend_table
from texas_hhcs.staffing import StaffingModel
from pathlib import Path

PROCESSED = Path('../data/processed')
REPORTS = Path('../reports')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

---
## 1. HCS Interest List Over Time

The HCS interest list is Texas's waiting list for community-based IDD services.
People wait **years** — sometimes over a decade — for a slot to open up.

Data compiled from LBB rider packets, HHSC Interest List Reduction reports, and news reporting.
Waitlist numbers are approximate due to periodic list cleanup/purges by HHSC.

**Note:** These figures represent the combined IDD waiver interest lists (HCS, TxHmL, CLASS, DBMD).
HCS is the largest component. Numbers are sourced from legislative testimony, HHSC reports,
and news coverage. Exact figures vary by reporting date within each biennium.

In [ ]:
# Historical HCS/IDD waiver interest list data
# Sources: LBB rider packets, HHSC Interest List Reduction reports, news coverage
# Wage assumptions from PFD rate methodology documents

budgets = [
    BienniumBudget(
        biennium="2014-15",
        legislature="83rd",
        hcs_slots_funded=1_750,
        hcs_waitlist_size=100_000,
        attendant_wage_assumption=10.60,
        source="LBB 83rd rider packet; HHSC testimony"
    ),
    BienniumBudget(
        biennium="2016-17",
        legislature="84th",
        hcs_slots_funded=1_375,
        hcs_waitlist_size=110_000,
        attendant_wage_assumption=10.60,
        source="LBB 84th rider packet; HHSC reports"
    ),
    BienniumBudget(
        biennium="2018-19",
        legislature="85th",
        hcs_slots_funded=1_600,
        hcs_waitlist_size=120_000,
        attendant_wage_assumption=10.60,
        source="LBB 85th rider packet; HHSC reports"
    ),
    BienniumBudget(
        biennium="2020-21",
        legislature="86th",
        hcs_slots_funded=3_000,
        hcs_waitlist_size=130_000,
        attendant_wage_assumption=10.60,
        source="LBB 86th rider packet; HHSC reports"
    ),
    BienniumBudget(
        biennium="2022-23",
        legislature="87th",
        hcs_slots_funded=2_800,
        hcs_waitlist_size=145_000,
        attendant_wage_assumption=10.60,
        source="LBB 87th rider packet; HHSC Interest List Reduction"
    ),
    BienniumBudget(
        biennium="2024-25",
        legislature="88th",
        hcs_slots_funded=3_292,
        hcs_waitlist_size=100_000,  # reduced after HHSC list cleanup
        hcs_strategy_spending=96_000_000,
        attendant_wage_assumption=10.60,
        source="LBB 88th rider packet; HHSC Interest List Reduction"
    ),
    BienniumBudget(
        biennium="2026-27",
        legislature="89th",
        hcs_slots_funded=None,  # pending appropriation
        hcs_waitlist_size=100_000,  # estimated
        attendant_wage_assumption=13.00,  # proposed new target
        source="KERA 2025-03-04; 89th session pending"
    ),
]

df_budget = pd.DataFrame(build_trend_table(budgets))
print("=== HCS Interest List & Slots Funded by Biennium ===\n")
print(f"{'Biennium':<12} {'Legislature':<12} {'Waitlist':>12} {'Slots Funded':>14} {'Wage Assump':>12} {'Source'}")
print('-' * 85)
for _, row in df_budget.iterrows():
    slots = f"{row['hcs_slots']:,.0f}" if pd.notna(row['hcs_slots']) else 'TBD'
    waitlist = f"{row['waitlist']:,.0f}" if pd.notna(row['waitlist']) else '—'
    wage = f"${row['wage_assumption']:.2f}" if pd.notna(row['wage_assumption']) else '—'
    print(f"{row['biennium']:<12} {row['legislature']:<12} {waitlist:>12} {slots:>14} {wage:>12}")

# Total slots funded over the period
total_slots = df_budget['hcs_slots'].sum()
print(f"\nTotal slots funded (2014-2025): {total_slots:,.0f}")
print(f"Current waitlist: ~100,000+")
print(f"At ~3,000 slots/biennium, clearing the list would take: {100_000 / 3_000 * 2:.0f} years")

In [ ]:
# Visualization: Waitlist vs Slots Funded
df_plot = df_budget[df_budget['hcs_slots'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# LEFT: Waitlist size over time (bar chart)
x_pos = np.arange(len(df_plot))
width = 0.35

bars1 = axes[0].bar(x_pos - width/2, df_plot['waitlist'] / 1000, width,
                     label='Interest List Size', color='#d32f2f', edgecolor='white')
bars2 = axes[0].bar(x_pos + width/2, df_plot['hcs_slots'] / 1000, width,
                     label='Slots Funded (biennium)', color='#388e3c', edgecolor='white')

axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(df_plot['biennium'], rotation=45, ha='right')
axes[0].set_ylabel('Thousands of People / Slots', fontsize=12)
axes[0].set_title('HCS Interest List vs. Slots Funded\nThe Gap That Never Closes',
                   fontweight='bold', fontsize=14)
axes[0].legend(fontsize=11)

# Add value labels
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.0f}K', ha='center', fontsize=10, fontweight='bold', color='#d32f2f')
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height()*1000:,.0f}', ha='center', fontsize=9, fontweight='bold', color='#388e3c')

# RIGHT: Years to clear the waitlist
current_waitlist = 100_000
slots_per_biennium = df_plot['hcs_slots'].mean()
slots_per_year = slots_per_biennium / 2

scenarios = {
    'Current pace\n(~1,500/yr)': slots_per_year,
    'Double funding\n(~3,000/yr)': slots_per_year * 2,
    'Triple funding\n(~4,500/yr)': slots_per_year * 3,
    '5x funding\n(~7,500/yr)': slots_per_year * 5,
}

years_to_clear = [current_waitlist / rate for rate in scenarios.values()]
colors_bar = ['#d32f2f', '#f57c00', '#1976d2', '#388e3c']

bars = axes[1].barh(list(scenarios.keys()), years_to_clear, color=colors_bar,
                     edgecolor='white', height=0.6)

for bar, yrs in zip(bars, years_to_clear):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{yrs:.0f} years', va='center', fontweight='bold', fontsize=12)

axes[1].set_xlabel('Years to Clear Waitlist', fontsize=12)
axes[1].set_title(f'How Long to Serve {current_waitlist:,} People\nat Various Funding Levels?',
                   fontweight='bold', fontsize=14)
axes[1].axvline(x=10, color='gray', linestyle=':', alpha=0.5)
axes[1].text(10.5, 0, 'A child born today\nwould be a teenager', fontsize=9, color='gray', va='bottom')

plt.tight_layout()
plt.savefig(REPORTS / 'waitlist_vs_slots.png', bbox_inches='tight')
plt.show()

print(f"\nAt current funding (~{slots_per_year:,.0f} slots/yr):")
print(f"  Time to clear 100,000-person waitlist: {current_waitlist/slots_per_year:.0f} years")
print(f"\nSaved: reports/waitlist_vs_slots.png")

---
## 2. The Workforce Crisis: Turnover Eats Everything

National DSP (Direct Support Professional) turnover rate: **~50%** annually.
This means half the workforce quits every year and must be replaced.

Sources:
- NCI Staff Stability Survey (2023): national DSP turnover rate 43.5%
- PHI (Paraprofessional Healthcare Institute): national HHA turnover 40-60%
- Pre-pandemic highs exceeded 50% in many states
- Texas likely tracks at or above the national average given its below-average wages

In [ ]:
# Turnover cost analysis for a 4-bed residential home
# Using conservative national estimates

TURNOVER_RATE = 0.50  # 50% annual DSP turnover (conservative for TX)
REPLACEMENT_COST = 2_500  # per hire: advertising, onboarding, training, productivity loss
FTES_PER_HOUSE = 5.0  # from StaffingModel (24/7 coverage)

turnovers_per_year = FTES_PER_HOUSE * TURNOVER_RATE
annual_turnover_cost = turnovers_per_year * REPLACEMENT_COST

print("=== Turnover Cost: 4-Bed Residential Home ===\n")
print(f"Staff needed (24/7 coverage):    {FTES_PER_HOUSE:.1f} FTEs")
print(f"Annual turnover rate:            {TURNOVER_RATE:.0%}")
print(f"Replacements per year:           {turnovers_per_year:.1f}")
print(f"Cost per replacement:            ${REPLACEMENT_COST:,.0f}")
print(f"Annual direct turnover cost:     ${annual_turnover_cost:,.0f}")
print()

# The FULL cost of turnover is much higher than just replacement
# Include: overtime to cover gaps, productivity loss, quality/safety incidents
overtime_coverage_cost = 5_000  # estimated: covering shifts during vacancies
quality_incident_cost = 3_000   # estimated: errors, missed meds, incident reports
total_turnover_cost = annual_turnover_cost + overtime_coverage_cost + quality_incident_cost

print(f"But the FULL cost of turnover includes:")
print(f"  Direct replacement (hiring, onboarding): ${annual_turnover_cost:,.0f}")
print(f"  Overtime to cover vacant shifts:          ${overtime_coverage_cost:,.0f}")
print(f"  Quality/safety incidents from new staff:  ${quality_incident_cost:,.0f}")
print(f"  TOTAL estimated turnover cost:            ${total_turnover_cost:,.0f}/yr")
print()

# What would a raise cost vs full turnover cost?
hours_per_fte = 2080
for raise_amt in [1.00, 2.00, 3.00]:
    cost_of_raise = FTES_PER_HOUSE * hours_per_fte * raise_amt
    net = cost_of_raise - total_turnover_cost
    print(f"  ${raise_amt:.0f}/hr raise for all staff: ${cost_of_raise:,.0f}/yr "
          f"({'saves' if net < 0 else 'costs'} ${abs(net):,.0f} vs turnover)")

# Scale to statewide
tx_hha = 314_610
est_houses = tx_hha / FTES_PER_HOUSE
statewide_turnover_cost = est_houses * total_turnover_cost
print(f"\n=== Statewide Estimate ===")
print(f"TX HHAs: {tx_hha:,.0f}")
print(f"Estimated 'house-equivalents': {est_houses:,.0f}")
print(f"Statewide annual FULL turnover cost: ${statewide_turnover_cost/1e6:,.0f}M")
print(f"\nTexas spends ~${statewide_turnover_cost/1e6:,.0f}M/yr on the CONSEQUENCES")
print(f"of low wages (replacement + overtime + quality issues) instead of")
print(f"investing that money in the workers themselves.")

---
## 3. The Causal Chain: Low Wages → Workforce Crisis → Access Crisis

This is not a coincidence. It's a system.

In [ ]:
# The Causal Chain — visualized as a flow diagram
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Box styling
box_props = dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black', linewidth=2)
arrow_props = dict(arrowstyle='->', color='#d32f2f', lw=3)

# Chain of causation (top to bottom)
chain = [
    (5, 9.2, 'HHSC sets $10.60/hr wage assumption\n(frozen since ~2014)', '#d32f2f'),
    (5, 7.8, 'Medicaid rates capped → providers\ncannot pay competitive wages', '#f57c00'),
    (5, 6.4, 'Workers leave for Buc-ee\'s ($18/hr),\nAmazon ($17.50/hr), Walmart ($14/hr)', '#1976d2'),
    (5, 5.0, '50% annual turnover → constant\nrehiring, training, understaffing', '#9c27b0'),
    (5, 3.6, 'Providers can\'t staff homes →\nstop accepting new clients', '#757575'),
    (5, 2.2, '100,000+ Texans wait YEARS\non the HCS interest list', '#d32f2f'),
]

for x, y, text, color in chain:
    ax.text(x, y, text, fontsize=13, fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.5', facecolor=color, edgecolor='black',
                      linewidth=2, alpha=0.15),
            color=color)

# Arrows between boxes
for i in range(len(chain) - 1):
    ax.annotate('', xy=(5, chain[i+1][1] + 0.55), xytext=(5, chain[i][1] - 0.55),
                arrowprops=arrow_props)

# Side annotations with data
annotations = [
    (1.2, 9.2, 'PFD Wage Calculator\ncell B7: $10.60\n(updated 2/7/2025)', 10),
    (8.8, 7.8, 'HCS breakeven: ~$15.50/hr\nICF breakeven: ~$15.00/hr', 10),
    (1.2, 6.4, 'BLS TX HHA mean: $12.19\nBut Buc-ee\'s pays $18.00', 10),
    (8.8, 5.0, 'NCI: 43.5% national turnover\n$6,250/yr per house in churn', 10),
    (1.2, 3.6, 'Houston Chronicle:\nprovider closures rising', 10),
    (8.8, 2.2, 'Texas Tribune (2024):\n"decade-long waits"', 10),
]

for x, y, text, fontsize in annotations:
    ax.text(x, y, text, fontsize=fontsize, ha='center', va='center',
            style='italic', color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                      edgecolor='#cccccc', alpha=0.8))

# Title
ax.text(5, 10.3, 'THE CAUSAL CHAIN', fontsize=18, fontweight='bold',
        ha='center', va='center', color='#d32f2f')
ax.text(5, 9.9, 'How a $10.60/hr Wage Assumption Creates a 100,000-Person Waitlist',
        fontsize=13, ha='center', va='center', color='#555555')

# Feedback loop arrow (right side)
ax.annotate('', xy=(9.5, 9.2), xytext=(9.5, 2.2),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5,
                            connectionstyle='arc3,rad=-0.15', linestyle='--'))
ax.text(10.0, 5.7, 'Feedback loop:\nshortage → overtime →\nburnout → more\nquits → worse\nshortage',
        fontsize=9, ha='center', va='center', color='gray', style='italic', rotation=0)

plt.tight_layout()
plt.savefig(REPORTS / 'causal_chain.png', bbox_inches='tight', dpi=150)
plt.show()

print("Saved: reports/causal_chain.png")

---
## 4. Key Findings

1. **100,000+ Texans** are on the HCS interest list — some waiting over a decade
2. **At current funding (~1,500 slots/yr), it would take ~67 years** to serve everyone waiting
3. **The wage assumption ($10.60) didn't move for 10+ years** while the waitlist grew from 100K to 145K
4. **50% annual turnover** costs each 4-bed house ~$14,250/yr in replacement, overtime, and quality costs
5. **A $1/hr raise is cheaper than the churn** — $10,400/yr vs $14,250/yr in full turnover costs
6. **Statewide turnover costs ~$897M/yr** — money spent on the CONSEQUENCES of low wages
7. **This is a vicious cycle**: low wages → turnover → understaffing → provider closures → longer waitlist